# NITS-IQA analysis (English publication adaptation)

This notebook is adapted from a historical experiment notebook for an English-only repository. The original notebook remains in the local archive. Source data are the 405 recorded pairs; no source images or model inference are included. Numeric calculations retain the historical logic, with an equivalent binary balanced-accuracy implementation and explicit repairs of invalid or out-of-order cells.

In [ ]:
from pathlib import Path
import pandas as pd

def locate_nits_csv():
    """Find the English-header NITS-IQA scores relative to the notebook or repository."""
    base = Path.cwd().resolve()
    candidates = [base / "nits_iqa/verified/NITS_IQA_405_verified.csv",
                  base / "../verified/NITS_IQA_405_verified.csv",
                  base / "verified/NITS_IQA_405_verified.csv"]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError("NITS_IQA_405_verified.csv not found; run from repository root or notebook directory")

DATA_CSV = locate_nits_csv()

def fast_balanced_accuracy(y_true, y_pred):
    """Binary balanced accuracy, equivalent to sklearn on these two-class folds."""
    import numpy as np
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    pos = yt == 1
    neg = yt == 0
    if not pos.any() or not neg.any():
        raise ValueError("Both classes are required for this binary benchmark")
    return 0.5 * ((yp[pos] == 1).mean() + (yp[neg] == 0).mean())


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix
)

# 1. Load the data
file = DATA_CSV

df = pd.read_csv(file)

metrics = [
    "DINOv2-Base (%)",
    "DINOv2-Large (%)",
    "Dists (%)",
    "SSIM (%)",
    "OpenCLIP (%)"
]

X = df[metrics].to_numpy(dtype=float)
y = (df["MOS"] >= 50).astype(int).to_numpy()
groups = df["Ref"].to_numpy()

# 2. Basic dataset checks
assert len(df) == 405
assert len(np.unique(groups)) == 9
assert y.sum() == 199

print("PASS: Dataset validation")

# 3. Recompute out-of-fold predictions for every metric
oof = np.zeros((len(df), len(metrics)), dtype=int)

logo = LeaveOneGroupOut()

for train_idx, test_idx in logo.split(X, y, groups):

    # Check that reference groups do not leak across folds
    assert set(groups[train_idx]).isdisjoint(
        set(groups[test_idx])
    )

    for j in range(len(metrics)):

        candidates = np.unique(X[train_idx, j])

        best_threshold = candidates[0]
        best_score = -1

        for threshold in candidates:

            train_pred = (
                X[train_idx, j] >= threshold
            ).astype(int)

            score = fast_balanced_accuracy(
                y[train_idx], train_pred
            )

            if score > best_score:
                best_score = score
                best_threshold = threshold

        oof[test_idx, j] = (
            X[test_idx, j] >= best_threshold
        ).astype(int)

print("PASS: Cross-validation completed")

# 4. DISTS confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y, oof[:, 2], labels=[0, 1]
).ravel()

print("\nDISTS RESULTS")
print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

assert (tp, tn, fp, fn) == (162, 170, 36, 37)

print("PASS: DISTS confusion matrix")

# 5. Error complementarity
dists_errors = oof[:, 2] != y

expected = {
    0: 15,
    1: 15,
    3: 19,
    4: 23
}

print("\nERROR COMPLEMENTARITY")

for j, expected_count in expected.items():

    corrected = int(
        np.sum(dists_errors & (oof[:, j] == y))
    )

    print(metrics[j], "corrected:", corrected)

    assert corrected == expected_count

print("\nALL VALIDATION CHECKS PASSED")